# Simple Widgets

A *widget* is an object whose attributes are tracked and can be modified externally.
Typically, these attributes can be modified by the user through the Jupyter UI.
With this library, they can also be modified by another user.
The [Yrs](https://github.com/y-crdt/y-crdt) library and its R bindings
[Yr](https://github.com/y-crdt/yr) are used to manage the widget state and manage
concurrent modifications via a `yr::Doc`.

To define a widget, we first need to create a class that will hold the attributes.
In the case of a Jupyter widget, we use `make_comm_widget`.
Comm widget are widgets that synchronize their state over a Jupyter Comm channel,
but all we need to know is that they are what should be used in Jupyter.

## A minimal slider
Let's create a slider class:

In [ ]:
IntSlider <- ywidgets::make_comm_widget(
    "IntSlider",
    value = 50L,
    min = 0L,
    max = 100L,
    step = 1L
)

And creating an instance of that class let us display the actual slider

In [ ]:
s <- IntSlider$new()
s

This class has some special functions called `mime_types` and `mime_bundle` that are use to tell Jupyter how to display it.
The slider wideget did not automatically acquire a UI.
These functions returned information that Jupyter could match with the [Yjs-widgets](https://github.com/QuantStack/yjs-widgets/) frontend extension to display something.
In particular, the class name `IntSlider` matches one of the few classes in Yjs-widget meant for testing that is available.
It handles the user interactions and communications with our R part of the widget

You can try to change the value of the slider in the UI above with your mouse, and revaluate the the cell below every time.
You will see a different value every time matching the slider UI.

In [ ]:
s$value

The other way around, if you modify the local R widget, you will see the slider UI change.

In [ ]:
s$value = 0

A given slider in the UI is tied to a specific object (not class) in R.
For instance creating a second object produce a similar looking slider in the UI, but its state value and visiual representation differ.
We can also take this opportunity to show that `new` can override the default values.

In [ ]:
s2 <- IntSlider$new(value = 2, max = 5)
s2

On the contrary, displaying the first widget object again shows the same slider value as the first one in the UI.
They *are* the same widget, and changing one will change all their representation.
For instance, mouving the slider below with your mouse will also modify the first one.

In [ ]:
s

## Another Text widget
This time we make another widget that displays a `textarea` to let a user freely enter text.

In [ ]:
Textarea <- ywidgets::make_comm_widget(
    "Textarea",
    value = "",
    rows = 0L,
    disabled = FALSE,
    continuous_update = TRUE
)

In [ ]:
t <- Textarea$new()
t

In [ ]:
t$value

In [ ]:
t$value <- "Hello world!"

## Reacting to user events
We can register a function to be called every time the user interacts with our slider.
This is useful to redraw a plot, or run some computation based on user-defined values.
To do this we use the `connect` method to compute the Fibonacci number of the slider's value.

In [ ]:
fibonacci <- function(n) {
  if (n <= 0) return(NA)
  if (n == 1) return(0)
  if (n == 2) return(1)
  a <- 0
  b <- 1
  for (i in 3:n) {
    temp <- a + b
    a <- b
    b <- temp
  }
  return(b)
}

Below, we use `value` as a parameter to `connect` to state that we want to listen to the changes on the `value` attribute.

In [ ]:
s3 <- IntSlider$new()
s3$connect(
    value = function(n) cat(paste0("fibonacci(", n, ") = ", fibonacci(n), "\n"))
)
s3

`value` is only a convention for `IntSlider` used in Yjs-widgets but does not hold special meaning for r-ywidgets.
We could also have listened on `min`, `max`, and `step`.

## Complex data type

More complex data types such as `Array` and `Map` can be tracked as widget attributes.
Because our widget library is backed by [CRDT](https://en.wikipedia.org/wiki/Conflict-free_replicated_data_type),
we need to use the CRDT types from [Yr](https://github.com/y-crdt/yr).

Under the hood, the integers used in the previous slider were transformed as an `Any` object.
The `IntSlider` definition is a convenience for:

In [ ]:
IntSlider <- ywidgets::make_comm_widget(
    "IntSlider",
    value = yr::Prelim$any(50L),
    min = yr::Prelim$any(0L),
    max = yr::Prelim$any(100L),
    step = yr::Prelim$any(1L)
)

`Prelim` stands for preliminary. It is used to create an object that will be inserted into a CRDT (such as the widget's root `yr::Doc`) and be explicit about how it should be handled.

Types (and their `Prelim`) that will be of particular interest are `Array`, `Map` and `Text`.
Let's use them to create a new widget.
Because we do not have a frontend representation for this widget, we will only interact with it via R.

In [ ]:
Dummy <- ywidgets::make_comm_widget(
    "Dummy",
    sequence = yr::Prelim$array(list(7, 8, 9)),
    number = yr::Prelim$any(33)
)
d <- Dummy$new()

The `Prelim` is converted to a CRDT and the `sequence` attribute is of type `ArrayRef`:

In [ ]:
d$sequence

To use the array methods such as `get`, `insert` *etc*, we need to use a slightly more complex syntax.
The reason for this is that to interact with the `yr::Doc` we need most of the time to pass a `Transaction`.
In short, this is a record that is used to tracks changes to the CRDT, but we don't really need to know more, simply how it interact with the widget API.

There are two way to get a transaction, `with_read` for readonly access and `with_write` for modifications.
These function will create a transaction and manage mandatory cleanup after it is no longer needed.
For instance to `get` the element at position `1`, we can use the following.

In [ ]:
sequence_elem_1 <- d$with_read(
    function(transaction) { d$sequence$get(transaction, 1) }
)
sequence_elem_1

Modifying the sequence is similar. Because arrays are also containers, we need to specify what types of value to insert.

In [ ]:
# Insert an element and retrieve the length
d$with_write(function(t) {
    d$sequence$insert(t, 1, yr::Prelim$any(33))
    d$sequence$len(t)
})

Previously, we were able to modify the slider value using a much simpler and familiar syntax.

```r
d$number <- 42
```

In reality, a transaction is also involved and managed automatically.
This syntaxic sugar is possible for whole value replacement, but not methods.
It is somehow equivalent to the following (fictive) code.

```r
d$with_write(function(t) {
    d$number$set(t, 42)  # set does not really exist
})
```

## Any vs CRDT

In the previous section, we have used `yr::Prelim$array` to create an `ArrayRef` but we did not mention that we can also create an array inside an `Any` (with `yr::Prelim$any(list(7, 8, 9)`).
Similarly, for a string, we can choose between a `Text` or a string stored inside an `Any`.

`Text`, `Array` and `Map` are the three main types that can be represented as an `Any` or a CRDT.
Let us now explain how to choose between one or the other.

With a rich CRDT `Text` object, everything happens as we are used to when collaborating a shared document online.
Modifications from both users are taken into account during the merge.

<table style="border-collapse:collapse;text-align:center;font-family:sans-serif;table-layout:fixed;width:480px">
  <tr>
    <th style="padding:8px 14px;border:1px solid #d0d7de;width:50%">User 1</th>
    <th style="padding:8px 14px;border:1px solid #d0d7de;width:50%">User 2</th>
  </tr>
  <tr>
    <td colspan="2" style="padding:8px 14px;border:1px solid #d0d7de;color:#8250df"><b>Synchronization</b></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul</code></td>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul</code></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de">
      <code>Hello Paul<span style="color:#1a7f37;font-weight:700">, how are you?</span></code><br>
      <small style="color:#1a7f37">insert ", how are you?" at end</small>
    </td>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul</code></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul<span style="color:#1a7f37;font-weight:700">, how are you?</span></code></td>
    <td style="padding:8px 14px;border:1px solid #d0d7de">
      <code><span style="color:#cf222e;text-decoration:line-through">Hello </span>Paul</code><br>
      <small style="color:#cf222e">delete "Hello " at start</small>
    </td>
  </tr>
  <tr>
    <td colspan="2" style="padding:8px 14px;border:1px solid #d0d7de;color:#8250df"><b>Synchronization</b></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Paul<span style="color:#1a7f37;font-weight:700">, how are you?</span></code></td>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Paul<span style="color:#1a7f37;font-weight:700">, how are you?</span></code></td>
  </tr>
</table>

On the contrary, using a string inside `Any`, the whole string is considered as a single entity that is replaced as a whole.
If multiple changes happen between synchronization, the last one will win.

<table style="border-collapse:collapse;text-align:center;font-family:sans-serif;table-layout:fixed;width:480px">
  <tr>
    <th style="padding:8px 14px;border:1px solid #d0d7de;width:50%">User 1</th>
    <th style="padding:8px 14px;border:1px solid #d0d7de;width:50%">User 2</th>
  </tr>
  <tr>
    <td colspan="2" style="padding:8px 14px;border:1px solid #d0d7de;color:#8250df"><b>Synchronization</b></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul</code></td>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul</code></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de">
      <code>Hello Paul<span style="color:#1a7f37;font-weight:700">, how are you?</span></code><br>
      <small style="color:#1a7f37">insert ", how are you?" at end</small>
    </td>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul</code></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Hello Paul<span style="color:#1a7f37;font-weight:700">, how are you?</span></code></td>
    <td style="padding:8px 14px;border:1px solid #d0d7de">
      <code><span style="color:#cf222e;text-decoration:line-through">Hello </span>Paul</code><br>
      <small style="color:#cf222e">delete "Hello " at start</small>
    </td>
  </tr>
  <tr>
    <td colspan="2" style="padding:8px 14px;border:1px solid #d0d7de;color:#8250df"><b>Synchronization</b></td>
  </tr>
  <tr>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Paul</code></td>
    <td style="padding:8px 14px;border:1px solid #d0d7de"><code>Paul</code></td>
  </tr>
</table>

So choosing between a rich `Text` or an `Any` string is a matter of whether we are interested in the changes that happen inside the string.
For something like a paragraph, `Text` is appropriate to merge all changes.
On the contrary, if we use a string to represent a single word like a tag, then inner changes do not matter. Whole string substitution is not only more appropriate but also has smaller footprint.